# Kathmandu Weather and Air Quality: Data Extraction

**Purpose:** download hourly weather and air quality data for Kathmandu from Open-Meteo and save it as one CSV file.

- Source 1: Open-Meteo Historical Weather API
- Source 2: Open-Meteo Air Quality API
- Location: Kathmandu only
- Period: 2022-08-05 to 2026-09-30, hourly
- Output: `kathmandu_weather_aq.csv`

## Step 1: Imports and settings

Import the libraries and set the location, date range, timezone and the two API addresses. Changing these values is the only edit needed to reuse the notebook.

In [1]:
import time
import requests
import pandas as pd

CITY = 'Kathmandu'
LAT, LON = 27.7172, 85.3240
START_DATE = '2022-08-05'
END_DATE = '2026-09-30'
TIMEZONE = 'Asia/Kathmandu'
OUTPUT_CSV = 'kathmandu_final.csv'

WEATHER_URL = 'https://archive-api.open-meteo.com/v1/archive'
AQ_URL = 'https://air-quality-api.open-meteo.com/v1/air-quality'

## Step 2: Choose the variables

List the weather variables and the air quality variables to request. The target for the project is `us_aqi`.

In [2]:
WEATHER_VARS = [
    'temperature_2m',
    'relative_humidity_2m',
    'dew_point_2m',
    'apparent_temperature',
    'precipitation',
    'rain',
    'weather_code',
    'pressure_msl',
    'cloud_cover',
    'wind_speed_10m',
    'wind_direction_10m',
    'wind_gusts_10m',
]

AQ_VARS = [
    'pm10',
    'pm2_5',
    'carbon_monoxide',
    'nitrogen_dioxide',
    'sulphur_dioxide',
    'ozone',
    'us_aqi',
]

## Step 3: Helper function to call the API

`fetch_hourly` sends one request to an Open-Meteo endpoint for a date range and returns a table with a `time` column. If the server is busy or rate limits the request, it waits and tries again.

In [3]:
def fetch_hourly(url, variables, start, end, retries=5):
    params = {
        'latitude': LAT,
        'longitude': LON,
        'start_date': start,
        'end_date': end,
        'hourly': ','.join(variables),
        'timezone': TIMEZONE,
    }
    for attempt in range(1, retries + 1):
        resp = requests.get(url, params=params, timeout=60)
        if resp.status_code == 200:
            df = pd.DataFrame(resp.json()['hourly'])
            df['time'] = pd.to_datetime(df['time'])
            return df
        wait = 5 * attempt
        print(f'  HTTP {resp.status_code} for {start}..{end}, retry {attempt}/{retries} in {wait}s')
        time.sleep(wait)
    raise RuntimeError(f'Failed to fetch {url} for {start}..{end}: {resp.text[:200]}')

## Step 4: Split the period into yearly chunks

Four years of hourly data in one request is large. Splitting by calendar year keeps each request small and easier to retry.

In [4]:
def year_chunks(start, end):
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    chunks = []
    for year in range(start.year, end.year + 1):
        s = max(start, pd.Timestamp(f'{year}-01-01'))
        e = min(end, pd.Timestamp(f'{year}-12-31'))
        chunks.append((s.strftime('%Y-%m-%d'), e.strftime('%Y-%m-%d')))
    return chunks

chunks = year_chunks(START_DATE, END_DATE)
chunks

[('2022-08-05', '2022-12-31'),
 ('2023-01-01', '2023-12-31'),
 ('2024-01-01', '2024-12-31'),
 ('2025-01-01', '2025-12-31'),
 ('2026-01-01', '2026-09-30')]

## Step 5: Download the weather data

Request the weather variables for every yearly chunk and stack the results into one table. A one second pause between requests keeps us within the free usage limits.

In [5]:
weather_parts = []
for s, e in chunks:
    print(f'Fetching weather {s} -> {e}')
    weather_parts.append(fetch_hourly(WEATHER_URL, WEATHER_VARS, s, e))
    time.sleep(1)

weather = pd.concat(weather_parts, ignore_index=True)
weather.shape

Fetching weather 2022-08-05 -> 2022-12-31
Fetching weather 2023-01-01 -> 2023-12-31
Fetching weather 2024-01-01 -> 2024-12-31
Fetching weather 2025-01-01 -> 2025-12-31
Fetching weather 2026-01-01 -> 2026-09-30


(36432, 13)

## Step 6: Download the air quality data

Same process for the air quality variables: pollutants and the US AQI.

In [6]:
aq_parts = []
for s, e in chunks:
    print(f'Fetching air quality {s} -> {e}')
    aq_parts.append(fetch_hourly(AQ_URL, AQ_VARS, s, e))
    time.sleep(1)

aq = pd.concat(aq_parts, ignore_index=True)
aq.shape

Fetching air quality 2022-08-05 -> 2022-12-31
Fetching air quality 2023-01-01 -> 2023-12-31
Fetching air quality 2024-01-01 -> 2024-12-31
Fetching air quality 2025-01-01 -> 2025-12-31
Fetching air quality 2026-01-01 -> 2026-09-30


(36432, 8)

## Step 7: Merge weather and air quality

Join the two tables on the hourly timestamp so each row has weather and air quality for the same hour. Rows without a `us_aqi` value are dropped because it is the target.

In [7]:
df = weather.merge(aq, on='time', how='inner')
df = df.dropna(subset=['us_aqi'])
df.shape

(36427, 20)

## Step 8: Add time columns

Extract hour, month, day of week (Monday = 0) and a weekend flag from the timestamp. Then remove any duplicate timestamps and sort in time order.

In [8]:
df['hour'] = df['time'].dt.hour
df['month'] = df['time'].dt.month
df['day_of_week'] = df['time'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

df = df.drop_duplicates(subset='time').sort_values('time').reset_index(drop=True)
df.head()

,time,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,weather_code,pressure_msl,cloud_cover,...,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,us_aqi,hour,month,day_of_week,is_weekend
0,2022-08-05 05:00:00,20.3,98,20.0,23.6,4.2,4.2,63,1008.6,100,...,22.2,478.0,14.1,5.2,68.0,63.0,5,8,4,0
1,2022-08-05 06:00:00,20.9,97,20.4,24.6,0.6,0.6,53,1009.2,99,...,20.8,435.0,11.7,4.5,75.0,63.0,6,8,4,0
2,2022-08-05 07:00:00,21.8,93,20.6,25.4,0.1,0.1,51,1009.4,89,...,17.7,374.0,8.2,3.6,85.0,64.0,7,8,4,0
3,2022-08-05 08:00:00,23.3,85,20.6,26.9,0.0,0.0,3,1008.7,93,...,16.2,303.0,4.4,2.5,97.0,65.0,8,8,4,0
4,2022-08-05 09:00:00,25.4,76,20.8,29.4,0.0,0.0,3,1007.5,100,...,14.2,272.0,2.9,2.1,103.0,65.0,9,8,4,0


## Step 9: Check the data

Confirm the size, the date range, rows per year, missing values and duplicates. These outputs are good evidence for the report.

In [9]:
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f"Range: {df['time'].min()} -> {df['time'].max()}")
print('Missing values:', int(df.isnull().sum().sum()))
print('Duplicate rows:', int(df.duplicated().sum()))
print()
print('Rows per year:')
print(df.groupby(df['time'].dt.year).size())

Shape: 36427 rows x 24 columns
Range: 2022-08-05 05:00:00 -> 2026-09-30 23:00:00
Missing values: 0
Duplicate rows: 0

Rows per year:
time
2022    3571
2023    8760
2024    8784
2025    8760
2026    6552
dtype: int64


## Step 10: Look at columns and types

List every column with its data type. This is the basis for the dataset description table in the report.

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 36427 entries, 0 to 36426
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   time                  36427 non-null  datetime64[us]
 1   temperature_2m        36427 non-null  float64       
 2   relative_humidity_2m  36427 non-null  int64         
 3   dew_point_2m          36427 non-null  float64       
 4   apparent_temperature  36427 non-null  float64       
 5   precipitation         36427 non-null  float64       
 6   rain                  36427 non-null  float64       
 7   weather_code          36427 non-null  int64         
 8   pressure_msl          36427 non-null  float64       
 9   cloud_cover           36427 non-null  int64         
 10  wind_speed_10m        36427 non-null  float64       
 11  wind_direction_10m    36427 non-null  int64         
 12  wind_gusts_10m        36427 non-null  float64       
 13  pm10                  36427

## Step 11: Save to CSV

Write the final table to a CSV file. This file is the dataset used in the rest of the project.

In [11]:
df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved to {OUTPUT_CSV}')

Saved to kathmandu_final.csv
